# Repeatable RLSE CUDA validation

This notebook calls `scripts/gpu_validation.py`; model execution, custom CUDA kernel validation (`scripts/validate_cuda_kernels.py`), and live HTTP server validation (`scripts/validate_gpu_server.py`) live in the repository. Every cell displays live `tqdm` progress bars so long builds, model downloads, and GPU tests never appear stuck.

### Option A: VS Code Local → Colab Runtime (Zero-Popup Fast Sync)
On your local checkout, run:
```bash
make colab-sync
# or: python3 scripts/gpu_validation.py pack --output /tmp/rlse-source.tar.gz --embed-notebook
```
Then run all cells below directly in VS Code connected to your Colab GPU runtime. Before committing, run `make colab-clear` (`python3 scripts/gpu_validation.py clear-notebook`) to strip the embedded archive.

### Option B: Browser Colab Upload
On your local checkout, run:
```bash
python3 scripts/gpu_validation.py pack --output /tmp/rlse-source.tar.gz
```
When `EMBEDDED_ARCHIVE_B64` is empty, Cell 2 prompts you to upload `/tmp/rlse-source.tar.gz`, `/tmp/rlse-source.tar.gz.json`, and `scripts/gpu_validation.py`.


In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
from tqdm.auto import tqdm

for bin_dir in (Path.home() / ".cargo/bin", Path("/usr/local/cuda/bin")):
    if bin_dir.is_dir() and str(bin_dir) not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = f"{bin_dir}{os.pathsep}{os.environ.get('PATH', '')}"

with tqdm(total=4, desc="Colab GPU & Rust Setup", dynamic_ncols=True) as setup_bar:
    setup_bar.set_postfix_str("Checking NVIDIA GPU (nvidia-smi)")
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total,driver_version", "--format=csv,noheader"], text=True, capture_output=True)
    if smi.returncode == 0:
        tqdm.write("GPU: " + smi.stdout.strip())
    setup_bar.update(1)

    setup_bar.set_postfix_str("Checking CUDA compiler (nvcc)")
    nvcc = subprocess.run(["nvcc", "--version"], text=True, capture_output=True)
    if nvcc.returncode == 0:
        tqdm.write(nvcc.stdout.strip().splitlines()[-1])
    setup_bar.update(1)

    if not all(shutil.which(tool) for tool in ("cargo", "rustc", "rustfmt", "cargo-clippy")):
        setup_bar.set_postfix_str("Installing Rust toolchain (rustup)")
        rust_bar = tqdm(desc="rustup install", unit="step", dynamic_ncols=True, leave=False)
        proc = subprocess.Popen(
            "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal -c rustfmt -c clippy",
            shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
        )
        for line in proc.stdout:
            s = line.strip()
            if s:
                rust_bar.update(1)
                rust_bar.set_postfix_str(s[:65])
        proc.wait()
        rust_bar.close()
        if proc.returncode != 0:
            raise RuntimeError(f"rustup failed with exit code {proc.returncode}")
        cargo_bin = str(Path.home() / ".cargo/bin")
        if cargo_bin not in os.environ.get("PATH", "").split(os.pathsep):
            os.environ["PATH"] = f"{cargo_bin}{os.pathsep}{os.environ.get('PATH', '')}"
    setup_bar.update(1)

    setup_bar.set_postfix_str("Verifying cargo & rustc")
    cargo_v = subprocess.check_output(["cargo", "--version"], text=True).strip()
    rustc_v = subprocess.check_output(["rustc", "--version"], text=True).strip()
    tqdm.write(f"Toolchain ready: {cargo_v} | {rustc_v}")
    setup_bar.update(1)


In [ ]:
import base64
import hashlib
import json
from pathlib import Path
import subprocess
import sys
import uuid
from tqdm.auto import tqdm

EMBEDDED_ARCHIVE_B64 = ""
EMBEDDED_METADATA_B64 = ""
EMBEDDED_SCRIPT_B64 = ""

required = {"rlse-source.tar.gz", "rlse-source.tar.gz.json", "gpu_validation.py"}
with tqdm(total=4, desc="Source Sync & Verification", dynamic_ncols=True) as sync_bar:
    sync_bar.set_postfix_str("Loading source archive payload")
    if EMBEDDED_ARCHIVE_B64 and EMBEDDED_METADATA_B64 and EMBEDDED_SCRIPT_B64:
        uploaded = {
            "rlse-source.tar.gz": base64.b64decode(EMBEDDED_ARCHIVE_B64),
            "rlse-source.tar.gz.json": base64.b64decode(EMBEDDED_METADATA_B64),
            "gpu_validation.py": base64.b64decode(EMBEDDED_SCRIPT_B64),
        }
        tqdm.write("Using embedded local-mode archive payload.")
    else:
        from google.colab import files
        uploaded = files.upload()
        if not required.issubset(uploaded):
            raise ValueError(f"Upload exactly named files: {sorted(required)}")
    sync_bar.update(1)

    sync_bar.set_postfix_str("Verifying bootstrap SHA-256")
    metadata = json.loads(uploaded["rlse-source.tar.gz.json"])
    bootstrap = uploaded["gpu_validation.py"]
    expected_script = metadata["source"]["files"]["scripts/gpu_validation.py"]["sha256"]
    if hashlib.sha256(bootstrap).hexdigest() != expected_script:
        raise ValueError("Bootstrap script does not match packaged source")
    sync_bar.update(1)

    sync_bar.set_postfix_str("Writing transport bundle")
    transport = Path("/content") / ("rlse-upload-" + uuid.uuid4().hex)
    transport.mkdir(parents=True)
    for name in required:
        (transport / name).write_bytes(uploaded[name])
    sync_bar.update(1)

    sync_bar.set_postfix_str("Unpacking & verifying source manifest")
    root = Path("/content") / ("rlse-" + uuid.uuid4().hex)
    subprocess.run([sys.executable, str(transport / "gpu_validation.py"), "unpack",
                    "--archive", str(transport / "rlse-source.tar.gz"),
                    "--sha256", metadata["archive_sha256"], "--destination", str(root)], check=True)
    harness = root / "scripts/gpu_validation.py"
    model_dir = Path("/content/rlse-models/Qwen3-0.6B")
    cache_dir = Path("/content/rlse-target")
    sync_bar.update(1)
    tqdm.write(f"Verified {len(metadata['source']['files'])} source files in {root}")


## Pinned pretrained artifacts

This downloads `Qwen/Qwen3-0.6B` into `/content/rlse-models/Qwen3-0.6B` at the revision pinned in the independent generation fixture (`c1899de289a04d12100db370d81485cdf75e47ca`). All four artifact hashes must match; subsequent runs in the same Colab session reuse the verified artifacts without re-downloading 1.2 GB.


In [ ]:
import os
import subprocess
import sys
from tqdm.auto import tqdm

dl_env = os.environ.copy()
dl_env["RLSE_STREAM_PROGRESS"] = "1"
proc = subprocess.Popen(
    [sys.executable, str(harness), "download-model", "--model-dir", str(model_dir)],
    env=dl_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
file_bar = tqdm(total=4, desc="Qwen3-0.6B Artifacts", dynamic_ncols=True)
byte_bar = None
current_name = None
for line in proc.stdout:
    s = line.strip()
    if not s:
        continue
    if s.startswith("[DOWNLOAD] "):
        _, name, done_s, total_s = s.split(" ", 3)
        done_b, total_b = int(done_s), int(total_s)
        if current_name != name:
            if byte_bar is not None:
                byte_bar.close()
            current_name = name
            byte_bar = tqdm(total=total_b or None, unit="B", unit_scale=True, unit_divisor=1024, desc=f"Downloading {name}", dynamic_ncols=True, leave=False)
        byte_bar.n = done_b
        byte_bar.refresh()
    else:
        if s.startswith("[") and ("] Verifying " in s):
            file_bar.update(1)
            file_bar.set_postfix_str(s.split("] ", 1)[-1])
        tqdm.write(s)
if byte_bar is not None:
    byte_bar.close()
file_bar.close()
proc.wait()
print("Artifact download exit code:", proc.returncode)


## CPU regressions, explicit CUDA inference, custom CUDA kernels, and live HTTP server validation

The harness records source/model identity, hardware, exact commands, exit codes, and logs. It runs:
1. Formatting (`cargo fmt --all --check`), CPU regressions (`cargo test --locked --all-targets`), and Python transport/evidence tests.
2. Pinned artifact hash verification.
3. CUDA Clippy and test builds (`--features cuda`).
4. Both explicit pretrained GPU gates (`test_gpu_inference` and `test_pretrained_quality`), requiring `ACTUAL_DEVICE=Cuda(...)` (pass `--include-numerics` to also run the slow CPU/F32 teacher-forced logit comparison).
5. Custom CUDA kernel verification (`scripts/validate_cuda_kernels.py`: F32 & F16 multi-row Fused RMSNorm across dims `128, 1024`, plus F32 & F16 Ragged Batched Tiled Online-Softmax Paged GQA Decode Attention across boundary lengths `1, 15, 16, 17, 31, 32, 33, 64, 128`, fragmented page tables, and shared prefix page reuse).
6. Live HTTP GPU server validation (`scripts/validate_gpu_server.py`: concurrent warm/cold prefix continuous batching and mid-stream SSE Ctrl-C graceful draining).


In [ ]:
import os
from tqdm.auto import tqdm

evidence = Path("/content") / ("rlse-evidence-" + uuid.uuid4().hex)
run_env = os.environ.copy()
run_env["RLSE_STREAM_PROGRESS"] = "1"
proc = subprocess.Popen([
    sys.executable, str(harness), "run",
    "--device", "cuda",
    "--model-dir", str(model_dir),
    "--cache-dir", str(cache_dir),
    "--include-kernels",
    "--include-server",
    "--output", str(evidence),
], env=run_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

stage_bar = tqdm(total=16, desc="Validation Pipeline", dynamic_ncols=True)
activity_bar = tqdm(desc="Current Step", unit="event", dynamic_ncols=True)
for line in proc.stdout:
    s = line.strip()
    if not s:
        continue
    if s.startswith("Running: "):
        cmd_short = s[len("Running: "):][:70]
        stage_bar.update(1)
        stage_bar.set_postfix_str(cmd_short)
        activity_bar.reset()
        activity_bar.set_description_str(f"Step {stage_bar.n}: {cmd_short[:40]}")
        tqdm.write(s)
    elif s.startswith("[LOG] "):
        msg = s[6:]
        activity_bar.update(1)
        activity_bar.set_postfix_str(msg[:68])
        if msg.startswith(("PASS:", "ACTUAL_DEVICE=", "test result:", "FAILED:", "BLOCKED:")):
            tqdm.write("  " + msg)
    elif s.startswith("[HEARTBEAT] "):
        activity_bar.set_postfix_str(s[12:][:68])
        activity_bar.refresh()
    else:
        tqdm.write(s)
proc.wait()
activity_bar.close()
stage_bar.close()

results = json.loads((evidence / "results.json").read_text())
print(json.dumps(results, indent=2))
try:
    from google.colab import files
    files.download(str(evidence) + ".tar.gz")
    files.download(str(evidence) + ".tar.gz.json")
except Exception as exc:
    print(f"Evidence bundle saved on runtime at {evidence}.tar.gz (browser download skipped: {exc})")
if proc.returncode:
    raise RuntimeError(f"Validation {results['status']}: {results.get('reason')}; inspect evidence")


## Follow-up: pretrained CUDA numerical comparison

Run this after the main pipeline passes, in the same connected runtime. It reuses
that run's verified source, pinned weights, and Cargo target cache without repeating
the full pipeline. It compares CPU/F32 and CUDA/F16 logits for prefill, restored
prefixes, and three teacher-forced decode steps, including batched versus single
decode. Existing tolerances and greedy-decision assertions remain unchanged.

This diagnostic is separate from the baseline result. Save its evidence even on
failure; a baseline pass does not imply numerical acceptance. CPU reference
execution may take several minutes. To include this gate in future full runs, add
`"--include-numerics",` to the main pipeline's argument list.


In [ ]:
import os
import re
import shutil
import time

baseline = json.loads((evidence / "results.json").read_text())
if baseline["status"] != "passed" or baseline["requested_device"] != "cuda":
    raise RuntimeError("Run the baseline CUDA pipeline successfully first")

numeric_evidence = Path("/content") / ("rlse-numerics-" + uuid.uuid4().hex)
numeric_evidence.mkdir()
# Preserve the baseline source, model, hardware, sampling, and CPU check records.
shutil.copytree(evidence, numeric_evidence / "baseline")
numeric_env = os.environ.copy()
numeric_env.update(RLSE_GPU_DEVICE="cuda", RLSE_GPU_MODEL_DIR=str(model_dir),
                   CARGO_TARGET_DIR=baseline["target_directory"])
command = ["cargo", "test", "--locked", "--features", "cuda",
           "--test", "test_pretrained_numerics", "--", "--ignored", "--nocapture"]
print("Running:", " ".join(command), flush=True)
started = time.monotonic()
proc = subprocess.Popen(command, cwd=root, env=numeric_env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
log_path = numeric_evidence / "numerics.log"
with log_path.open("w") as log:
    for line in proc.stdout:
        log.write(line)
        log.flush()
        print(line, end="", flush=True)
proc.wait()
content = log_path.read_text()
passed = (proc.returncode == 0 and "ACTUAL_DEVICE=Cuda(" in content
          and re.search(r"test result: ok\. 1 passed; 0 failed; 0 ignored;", content) is not None)
summary = {"status": "passed" if passed else "failed", "argv": command,
           "cwd": str(root), "requested_device": "cuda", "exit_code": proc.returncode,
           "seconds": round(time.monotonic() - started, 3),
           "actual_device_markers": [line for line in content.splitlines() if "ACTUAL_DEVICE=" in line],
           "environment": {key: numeric_env[key] for key in
                           ("RLSE_GPU_DEVICE", "RLSE_GPU_MODEL_DIR", "CARGO_TARGET_DIR")},
           "reference": "CPU/F32 versus CUDA/F16; greedy; 3 teacher-forced decode steps",
           "limitations": ["Diagnostic only; no performance claim. See baseline for source/model identity."]}
(numeric_evidence / "results.json").write_text(json.dumps(summary, indent=2) + "\n")
bundle = shutil.make_archive(str(numeric_evidence), "gztar", root_dir=numeric_evidence)
print(json.dumps(summary, indent=2))
print("Evidence:", bundle)
try:
    from google.colab import files
    files.download(bundle)
except Exception as exc:
    print(f"Download skipped ({exc}); retrieve {bundle} from the runtime")
if not passed:
    raise RuntimeError("CUDA numerical comparison failed; preserve numerics.log and the evidence bundle")
